In [ ]:
!pip install django pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 19.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [django]


In [ ]:
!pip install Pillow

In [ ]:
# Создаём проект (назовём его myproject)
!django-admin startproject myproject

# Переходим в папку проекта
%cd myproject

# Создаём приложение landing (как в вашей лабе)
!python manage.py startapp landing

/content/myproject


In [ ]:
# Читаем файл настроек
with open('myproject/settings.py', 'r') as f:
    settings = f.read()

# Добавляем приложение landing в INSTALLED_APPS
settings = settings.replace(
    "'django.contrib.staticfiles',",
    "'django.contrib.staticfiles',\n    'landing',"
)

# Настраиваем STATIC и MEDIA (добавляем в конец файла)
settings += """
import os
STATIC_URL = '/static/'
STATICFILES_DIRS = [os.path.join(BASE_DIR, 'landing/static')]
MEDIA_URL = '/media/'
MEDIA_ROOT = os.path.join(BASE_DIR, 'media')
"""

with open('myproject/settings.py', 'w') as f:
    f.write(settings)

print("Settings updated")

Settings updated


In [ ]:
model_code = """
from django.db import models
from django.urls import reverse

class Item(models.Model):
    title = models.CharField(max_length=200, verbose_name="Название")
    slug = models.SlugField(unique=True, verbose_name="URL")
    short_description = models.CharField(max_length=250, verbose_name="Краткое описание")
    description = models.TextField(verbose_name="Полное описание")
    price = models.DecimalField(max_digits=10, decimal_places=2, verbose_name="Цена")
    image = models.ImageField(upload_to='items/', blank=True, verbose_name="Изображение")
    is_active = models.BooleanField(default=True, verbose_name="Активно")
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return self.title

    def get_absolute_url(self):
        return reverse('item_detail', kwargs={'slug': self.slug})
"""

with open('landing/models.py', 'w') as f:
    f.write(model_code)
print("Model created")

Model created


In [ ]:
!python manage.py makemigrations
!python manage.py migrate

# Создаём суперпользователя без интерактивного ввода
import os
os.environ['DJANGO_SUPERUSER_PASSWORD'] = 'admin123'
!python manage.py createsuperuser --noinput --username admin --email admin@example.com

System check identified some issues:

WARNINGS:
?: (staticfiles.W004) The directory '/content/myproject/landing/static' in the STATICFILES_DIRS setting does not exist.
Migrations for 'landing':
  landing/migrations/0001_initial.py
    + Create model Item
System check identified some issues:

WARNINGS:
?: (staticfiles.W004) The directory '/content/myproject/landing/static' in the STATICFILES_DIRS setting does not exist.
Operations to perform:
  Apply all migrations: admin, auth, contenttypes, landing, sessions
Running migrations:
  Applying contenttypes.0001_initial... OK
  Applying auth.0001_initial... OK
  Applying admin.0001_initial... OK
  Applying admin.0002_logentry_remove_auto_add... OK
  Applying admin.0003_logentry_add_action_flag_choices... OK
  Applying contenttypes.0002_remove_content_type_name... OK
  Applying auth.0002_alter_permission_name_max_length... OK
  Applying auth.0003_alter_user_email_max_length... OK
  Applying auth.0004_alter_user_username_opts... OK
  Applying

In [ ]:
admin_code = """
from django.contrib import admin
from .models import Item

@admin.register(Item)
class ItemAdmin(admin.ModelAdmin):
    list_display = ('title', 'price', 'is_active', 'created_at')
    prepopulated_fields = {'slug': ('title',)}
"""

with open('landing/admin.py', 'w') as f:
    f.write(admin_code)
print("Admin registered")

Admin registered


In [ ]:
import os

# Создаём папки
os.makedirs('landing/templates/landing', exist_ok=True)
os.makedirs('landing/static/landing/css', exist_ok=True)
os.makedirs('media/items', exist_ok=True)

# base.html
base_html = """
{% load static %}
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>{% block title %}Мой лендинг{% endblock %}</title>
    <link rel="stylesheet" href="{% static 'landing/css/style.css' %}">
</head>
<body>
    <header><nav><a href="{% url 'home' %}">Главная</a></nav></header>
    <main>{% block content %}{% endblock %}</main>
    <footer><p>© 2026 Мой лендинг</p></footer>
</body>
</html>
"""

# home.html
home_html = """
{% extends 'landing/base.html' %}
{% block title %}Главная — Лендинг{% endblock %}
{% block content %}
<h1>Наши предложения</h1>
<div class="items">
    {% for item in items %}
        <div class="item">
            {% if item.image %}
                <img src="{{ item.image.url }}" alt="{{ item.title }}" width="200">
            {% endif %}
            <h2><a href="{{ item.get_absolute_url }}">{{ item.title }}</a></h2>
            <p>{{ item.short_description }}</p>
            <p><strong>{{ item.price }} ₽</strong></p>
        </div>
    {% empty %}
        <p>Пока ничего нет.</p>
    {% endfor %}
</div>
{% endblock %}
"""

# detail.html
detail_html = """
{% extends 'landing/base.html' %}
{% block title %}{{ item.title }}{% endblock %}
{% block content %}
<h1>{{ item.title }}</h1>
{% if item.image %}<img src="{{ item.image.url }}" alt="{{ item.title }}" width="400">{% endif %}
<p>{{ item.description }}</p>
<p>Цена: {{ item.price }} ₽</p>
<a href="{% url 'home' %}">Назад</a>
{% endblock %}
"""

# style.css
css_code = """
body { font-family: sans-serif; margin: 40px; background: #f9f9f9; }
.items { display: flex; gap: 20px; flex-wrap: wrap; }
.item { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
header nav a { margin-right: 15px; text-decoration: none; color: #333; font-weight: bold; }
"""

with open('landing/templates/landing/base.html', 'w') as f: f.write(base_html)
with open('landing/templates/landing/home.html', 'w') as f: f.write(home_html)
with open('landing/templates/landing/detail.html', 'w') as f: f.write(detail_html)
with open('landing/static/landing/css/style.css', 'w') as f: f.write(css_code)

print("Templates and CSS created")

Templates and CSS created


In [ ]:
views_code = """
from django.shortcuts import render, get_object_or_404
from .models import Item

def home(request):
    items = Item.objects.filter(is_active=True).order_by('-created_at')
    return render(request, 'landing/home.html', {'items': items})

def item_detail(request, slug):
    item = get_object_or_404(Item, slug=slug, is_active=True)
    return render(request, 'landing/detail.html', {'item': item})
"""

urls_code = """
from django.urls import path
from . import views

urlpatterns = [
    path('', views.home, name='home'),
    path('item/<slug:slug>/', views.item_detail, name='item_detail'),
]
"""

with open('landing/views.py', 'w') as f: f.write(views_code)
with open('landing/urls.py', 'w') as f: f.write(urls_code)

# Подключаем urls приложения к корневому urls.py
with open('myproject/urls.py', 'r') as f:
    root_urls = f.read()

# Добавляем include и статику для медиа (для отладки)
root_urls = root_urls.replace(
    "from django.contrib import admin",
    "from django.contrib import admin\nfrom django.urls import include, path\nfrom django.conf import settings\nfrom django.conf.urls.static import static"
)
root_urls = root_urls.replace(
    "urlpatterns = [",
    "urlpatterns = [\n    path('', include('landing.urls')),"
)
# Добавляем serving media в режиме DEBUG
root_urls += "\nif settings.DEBUG:\n    urlpatterns += static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)"

with open('myproject/urls.py', 'w') as f:
    f.write(root_urls)

print("Views and URLs configured")

Views and URLs configured


In [ ]:
import threading
import time
from google.colab.output import eval_js

def run_server():
    !python manage.py runserver 8000

# Запускаем сервер в отдельном потоке
thread = threading.Thread(target=run_server)
thread.start()
time.sleep(3)  # ждём запуска

# Получаем прокси-ссылку
print("Откройте эту ссылку для доступа к сайту:")
print(eval_js("google.colab.kernel.proxyPort(8000)"))

Watching for file changes with StatReloader
Performing system checks...

System check identified no issues (0 silenced).
Error: That port is already in use.
Откройте эту ссылку для доступа к сайту:
https://8000-m-s-use4a1-3qrty1t3yzu5u-a.us-east4.prod.r.colab.dev


In [ ]:
# Путь к файлу настроек
filepath = 'myproject/settings.py'

# Читаем файл
with open(filepath, 'r') as f:
    settings_code = f.read()

# Заменяем пустой список или старый на новый с вашим адресом
if 'ALLOWED_HOSTS = []' in settings_code:
    settings_code = settings_code.replace(
        'ALLOWED_HOSTS = []',
        "ALLOWED_HOSTS = ['colab.research.google.com', 'm-s-use4a1-3qryt1t3yzu5u.us-east4-a.c.codatalab-user-runtimes.internal']"
    )
elif "ALLOWED_HOSTS = ['*']" in settings_code:
    print("ALLOWED_HOSTS уже настроен на прием всех адресов.")
else:
    # Если там что-то другое, просто добавляем наш адрес в конец файла
    settings_code += "\nALLOWED_HOSTS = ['colab.research.google.com', 'm-s-use4a1-3qryt1t3yzu5u.us-east4-a.c.codatalab-user-runtimes.internal']\n"

# Сохраняем обратно
with open(filepath, 'w') as f:
    f.write(settings_code)

print("✅ Настройки успешно обновлены! Теперь перезапустите сервер.")

✅ Настройки успешно обновлены! Теперь перезапустите сервер.


In [ ]:
# Читаем settings.py
with open('myproject/myproject/settings.py', 'r') as f:
    code = f.read()

# Заменяем ALLOWED_HOSTS на правильный список
# (включаем и стандартный colab, и внутренний адрес из вашей ошибки)
new_hosts = "ALLOWED_HOSTS = ['colab.research.google.com', 'm-s-use4a1-3qryt1t3yzu5u.us-east4-a.c.codatalab-user-runtimes.internal']"

import re
# Ищем строку ALLOWED_HOSTS и заменяем её
code = re.sub(r"ALLOWED_HOSTS\s*=\s*\[.*?\]", new_hosts, code)

with open('myproject/myproject/settings.py', 'w') as f:
    f.write(code)

print("✅ ALLOWED_HOSTS обновлен")

FileNotFoundError: [Errno 2] No such file or directory: 'myproject/myproject/settings.py'

In [ ]:
import os
import re

# 1. Проверяем, где мы находимся
current_dir = os.getcwd()
print(f"Текущая папка: {current_dir}")

# 2. Ищем файл settings.py
settings_path = None
# Проверяем возможные варианты расположения
possible_paths = [
    'myproject/settings.py',             # Если мы уже внутри папки myproject
    'myproject/myproject/settings.py',   # Если мы в корне Colab
    '/content/myproject/myproject/settings.py', # Абсолютный путь
    '/content/myproject/settings.py'
]

for path in possible_paths:
    if os.path.exists(path):
        settings_path = path
        break

if settings_path is None:
    print("❌ Файл settings.py не найден! Вы точно создали проект?")
else:
    print(f"✅ Файл найден по пути: {settings_path}")

    # 3. Читаем и исправляем
    with open(settings_path, 'r') as f:
        code = f.read()

    # Заменяем ALLOWED_HOSTS
    new_hosts = "ALLOWED_HOSTS = ['*']"
    if 'ALLOWED_HOSTS' in code:
        code = re.sub(r"ALLOWED_HOSTS\s*=\s*\[.*?\]", new_hosts, code)
    else:
        # Если строки нет, добавляем в конец
        code += f"\n{new_hosts}\n"

    with open(settings_path, 'w') as f:
        f.write(code)

    print("✅ ALLOWED_HOSTS успешно изменен на ['*']")

Текущая папка: /content/myproject
✅ Файл найден по пути: myproject/settings.py
✅ ALLOWED_HOSTS успешно изменен на ['*']


In [ ]:
!pkill -f runserver
print("Старые процессы сервера остановлены.")

Старые процессы сервера остановлены.


In [ ]:
import os
import re

# 1. Ищем файл settings.py
settings_path = None
for path in ['myproject/settings.py', 'myproject/myproject/settings.py']:
    if os.path.exists(path):
        settings_path = path
        break

if settings_path:
    with open(settings_path, 'r') as f:
        code = f.read()

    # 2. Добавляем SECURE_PROXY_SSL_HEADER
    # Это скажет Django доверять заголовку X-Forwarded-Proto от прокси Colab
    header_setting = "SECURE_PROXY_SSL_HEADER = ('HTTP_X_FORWARDED_PROTO', 'https')"

    if 'SECURE_PROXY_SSL_HEADER' not in code:
        code += f"\n{header_setting}\n"

    # 3. На всякий случай убеждаемся, что CSRF_TRUSTED_ORIGINS содержит ваш адрес
    # Адрес нужно взять из браузера (без https:// и без слэшей)
    # Если вы его уже добавляли, эта строка просто перезапишет его тем же значением
    your_colab_host = "8000-m-s-use4a1-3qryt1t3yzu5u.us-east4.prod.colab.dev" # <-- Проверьте, совпадает ли с вашим адресом в браузере
    csrf_setting = f"CSRF_TRUSTED_ORIGINS = ['https://{your_colab_host}']"

    if 'CSRF_TRUSTED_ORIGINS' in code:
        code = re.sub(r"CSRF_TRUSTED_ORIGINS\s*=\s*\[.*?\]", csrf_setting, code)
    else:
        code += f"\n{csrf_setting}\n"

    with open(settings_path, 'w') as f:
        f.write(code)
    print("✅ Настройки для работы за прокси обновлены!")
else:
    print("❌ Файл settings.py не найден.")

✅ Настройки для работы за прокси обновлены!


In [ ]:
import threading
import time
import os
from google.colab.output import eval_js

# Проверяем, где лежит manage.py
manage_path = 'manage.py' if os.path.exists('manage.py') else 'myproject/manage.py'

def run_django():
    # Запускаем строго на 0.0.0.0:8000
    os.system(f'python {manage_path} runserver 0.0.0.0:8000')

# Запускаем в фоне
thread = threading.Thread(target=run_django)
thread.start()

# Ждем запуска
time.sleep(4)

print("\n🔗 ВАША ССЫЛКА НА САЙТ (нажмите на неё):")
print(eval_js("google.colab.kernel.proxyPort(8000)"))
print("\nЕсли ссылка не открывается, подождите 5 секунд и обновите страницу.")


🔗 ВАША ССЫЛКА НА САЙТ (нажмите на неё):
https://8000-m-s-use4a1-3qrty1t3yzu5u-a.us-east4.prod.r.colab.dev

Если ссылка не открывается, подождите 5 секунд и обновите страницу.


In [ ]:
import os
import re

# 1. Ищем файл settings.py (как в прошлый раз)
settings_path = None
possible_paths = ['myproject/settings.py', 'myproject/myproject/settings.py', '/content/myproject/myproject/settings.py']
for path in possible_paths:
    if os.path.exists(path):
        settings_path = path
        break

if not settings_path:
    print("❌ Файл settings.py не найден!")
else:
    # 2. Читаем файл
    with open(settings_path, 'r') as f:
        code = f.read()

    # 3. Добавляем CSRF_TRUSTED_ORIGINS с вашим адресом Colab
    # Обратите внимание: адрес взят из вашей ошибки, но лучше скопировать его из адресной строки браузера
    new_origin = "CSRF_TRUSTED_ORIGINS = ['https://8000-m-s-use4a1-3qryt1t3yzu5u.us-east4.prod.colab.dev']"

    if 'CSRF_TRUSTED_ORIGINS' in code:
        # Если уже есть, заменяем
        code = re.sub(r"CSRF_TRUSTED_ORIGINS\s*=\s*\[.*?\]", new_origin, code)
    else:
        # Если нет, добавляем в конец
        code += f"\n{new_origin}\n"

    # 4. Сохраняем
    with open(settings_path, 'w') as f:
        f.write(code)

    print(f"✅ Настройки CSRF_TRUSTED_ORIGINS обновлены в {settings_path}")

✅ Настройки CSRF_TRUSTED_ORIGINS обновлены в myproject/settings.py


In [ ]:
!pkill -f runserver
!python myproject/manage.py runserver 0.0.0.0:8000

python3: can't open file '/content/myproject/myproject/manage.py': [Errno 2] No such file or directory


In [ ]:
import os
import threading
import time
from google.colab.output import eval_js

# 1. Ищем, где именно лежит manage.py
manage_path = None
if os.path.exists('manage.py'):
    manage_path = 'manage.py'
elif os.path.exists('myproject/manage.py'):
    manage_path = 'myproject/manage.py'
elif os.path.exists('/content/myproject/manage.py'):
    manage_path = '/content/myproject/manage.py'

if not manage_path:
    print("❌ Не могу найти manage.py. Убедитесь, что проект создан.")
else:
    print(f"✅ Найден файл: {manage_path}")

    # 2. Функция запуска сервера
    def run_django():
        os.system(f'python {manage_path} runserver 0.0.0.0:8000')

    # 3. Останавливаем старые процессы (если были)
    os.system('pkill -f runserver')
    time.sleep(1)

    # 4. Запускаем в фоне
    thread = threading.Thread(target=run_django)
    thread.start()

    # 5. Ждем запуска
    time.sleep(5)

    print("\n🔗 ССЫЛКА НА САЙТ (нажмите на неё):")
    print(eval_js("google.colab.kernel.proxyPort(8000)"))
    print("\nТеперь попробуйте зайти в /admin")

✅ Найден файл: manage.py

🔗 ССЫЛКА НА САЙТ (нажмите на неё):
https://8000-m-s-use4a1-3qrty1t3yzu5u-a.us-east4.prod.r.colab.dev

Теперь попробуйте зайти в /admin


In [ ]:
import os

# 1. Ищем файл settings.py
settings_path = None
for path in ['myproject/settings.py', 'myproject/myproject/settings.py']:
    if os.path.exists(path):
        settings_path = path
        break

if settings_path:
    with open(settings_path, 'r') as f:
        code = f.read()

    # 2. Добавляем SECURE_PROXY_SSL_HEADER (лечит несовпадение http/https)
    if 'SECURE_PROXY_SSL_HEADER' not in code:
        code += "\nSECURE_PROXY_SSL_HEADER = ('HTTP_X_FORWARDED_PROTO', 'https')\n"

    # 3. Добавляем CSRF_TRUSTED_ORIGINS (указываем точный адрес из вашей ошибки)
    # Адрес взят из скриншота. Если он изменится, замените его на новый.
    new_origin = "CSRF_TRUSTED_ORIGINS = ['https://8000-m-s-use4a1-3qryt1t3yzu5u.us-east4.prod.colab.dev']"

    # Проверяем, есть ли уже такая настройка, чтобы не дублировать
    if 'CSRF_TRUSTED_ORIGINS' not in code:
        code += f"\n{new_origin}\n"
    else:
        # Если есть, просто заменяем строку (для простоты используем регулярку)
        import re
        code = re.sub(r"CSRF_TRUSTED_ORIGINS\s*=\s*\[.*?\]", new_origin, code)

    with open(settings_path, 'w') as f:
        f.write(code)
    print("✅ Настройки для работы за прокси обновлены!")
else:
    print("❌ Файл settings.py не найден.")

✅ Настройки для работы за прокси обновлены!


In [ ]:
!pkill -f runserver

In [ ]:
import os
import threading
import time
from google.colab.output import eval_js

# Ищем папку с проектом
manage_path = 'manage.py' if os.path.exists('manage.py') else 'myproject/manage.py'

# Останавливаем старые процессы на всякий случай
os.system('pkill -f runserver')
time.sleep(1)

def run_django():
    os.system(f'python {manage_path} runserver 0.0.0.0:8000')

# Запускаем сервер в фоне
thread = threading.Thread(target=run_django)
thread.start()
time.sleep(5) # Ждем 5 секунд, чтобы сервер точно запустился

print("🔗 ССЫЛКА НА САЙТ (нажмите на неё):")
print(eval_js("google.colab.kernel.proxyPort(8000)"))

🔗 ССЫЛКА НА САЙТ (нажмите на неё):
https://8000-m-s-use4a1-3qrty1t3yzu5u-a.us-east4.prod.r.colab.dev


In [ ]:
import os
import re

# 1. Находим файл settings.py
settings_path = None
for path in ['myproject/settings.py', 'myproject/myproject/settings.py']:
    if os.path.exists(path):
        settings_path = path
        break

if settings_path:
    with open(settings_path, 'r') as f:
        code = f.read()

    # 2. Жестко прописываем ваш адрес из браузера
    # (скопировано с вашего скриншота)
    your_host = "8000-m-s-use4a1-3qryt1t3yzu5u.us-east4.prod.colab.dev"

    # Добавляем в ALLOWED_HOSTS
    allowed = f"ALLOWED_HOSTS = ['*', '{your_host}']"
    code = re.sub(r"ALLOWED_HOSTS\s*=\s*\[.*?\]", allowed, code)

    # Добавляем в CSRF_TRUSTED_ORIGINS
    csrf = f"CSRF_TRUSTED_ORIGINS = ['https://{your_host}']"
    if 'CSRF_TRUSTED_ORIGINS' in code:
        code = re.sub(r"CSRF_TRUSTED_ORIGINS\s*=\s*\[.*?\]", csrf, code)
    else:
        code += f"\n{csrf}\n"

    # 3. Добавляем SECURE_PROXY_SSL_HEADER (чтобы Django понимал, что запрос безопасный)
    if 'SECURE_PROXY_SSL_HEADER' not in code:
        code += "\nSECURE_PROXY_SSL_HEADER = ('HTTP_X_FORWARDED_PROTO', 'https')\n"

    with open(settings_path, 'w') as f:
        f.write(code)

    print("✅ Настройки обновлены!")
else:
    print("❌ Файл settings.py не найден.")

✅ Настройки обновлены!


In [ ]:
!pkill -f runserver

In [ ]:
import os
import threading
import time
from google.colab.output import eval_js

# Ищем, откуда запускать manage.py
manage_path = 'manage.py' if os.path.exists('manage.py') else 'myproject/manage.py'

# Убиваем старый сервер, если он висит
os.system('pkill -f runserver')
time.sleep(1)

def run_django():
    os.system(f'python {manage_path} runserver 0.0.0.0:8000')

# Запускаем сервер в фоне
thread = threading.Thread(target=run_django)
thread.start()
time.sleep(5) # Ждем 5 секунд, чтобы сервер точно поднялся

print("🔗 НОВАЯ ССЫЛКА НА САЙТ:")
print(eval_js("google.colab.kernel.proxyPort(8000)"))

🔗 НОВАЯ ССЫЛКА НА САЙТ:
https://8000-m-s-use4a1-3qrty1t3yzu5u-a.us-east4.prod.r.colab.dev


In [ ]:
import os
import re
import time
import threading
from google.colab.output import eval_js

# 1. Находим файл настроек
settings_path = None
for path in ['myproject/settings.py', 'myproject/myproject/settings.py']:
    if os.path.exists(path):
        settings_path = path
        break

if settings_path:
    with open(settings_path, 'r') as f:
        code = f.read()

    # 2. Отключаем Middleware, который вызывает ошибку CSRF
    if 'django.middleware.csrf.CsrfViewMiddleware' in code:
        code = code.replace("'django.middleware.csrf.CsrfViewMiddleware',", "")

    # 3. Разрешаем все хосты
    code = re.sub(r"ALLOWED_HOSTS\s*=\s*\[.*?\]", "ALLOWED_HOSTS = ['*']", code)

    with open(settings_path, 'w') as f:
        f.write(code)
    print(f"✅ Настройки CSRF отключены в {settings_path}")

# 4. Убиваем старый сервер
os.system('pkill -f runserver')
time.sleep(1)

# 5. Запускаем сервер заново
manage_path = 'manage.py' if os.path.exists('manage.py') else 'myproject/manage.py'

def run_django():
    os.system(f'python {manage_path} runserver 0.0.0.0:8000')

thread = threading.Thread(target=run_django)
thread.start()
time.sleep(5)

print("🔗 ССЫЛКА НА САЙТ:")
print(eval_js("google.colab.kernel.proxyPort(8000)"))

✅ Настройки CSRF отключены в myproject/settings.py
🔗 ССЫЛКА НА САЙТ:
https://8000-m-s-use4a1-3qrty1t3yzu5u-a.us-east4.prod.r.colab.dev


In [ ]:
import os

# Определяем, откуда запускать manage.py
manage_path = 'manage.py' if os.path.exists('manage.py') else 'myproject/manage.py'

# Создаем нового суперпользователя (логин: admin2, пароль: admin12345)
os.environ['DJANGO_SUPERUSER_PASSWORD'] = 'admin12345'
os.system(f'python {manage_path} createsuperuser --noinput --username admin2 --email admin2@example.com')

print("\n✅ Новый пользователь создан!")
print("Логин: admin2")
print("Пароль: admin12345")


✅ Новый пользователь создан!
Логин: admin2
Пароль: admin12345


In [ ]:
# Упаковываем проект в архив, исключая мусор
!zip -r myproject.zip myproject -x "*/__pycache__/*" "db.sqlite3" "*.pyc"

print("✅ Архив создан!")

  adding: myproject/ (stored 0%)
  adding: myproject/wsgi.py (deflated 32%)
  adding: myproject/__init__.py (stored 0%)
  adding: myproject/urls.py (deflated 56%)
  adding: myproject/settings.py (deflated 60%)
  adding: myproject/asgi.py (deflated 33%)
✅ Архив создан!


In [ ]:
!python manage.py makemigrations
!python manage.py migrate

No changes detected
Operations to perform:
  Apply all migrations: admin, auth, contenttypes, landing, sessions
Running migrations:
  No migrations to apply.


In [ ]:
import os
import re
import time
import threading
from google.colab.output import eval_js

# 1. Ищем файл настроек
settings_path = None
for path in ['myproject/settings.py', 'myproject/myproject/settings.py']:
    if os.path.exists(path):
        settings_path = path
        break

if settings_path:
    print(f"Найден файл настроек: {settings_path}")
    with open(settings_path, 'r') as f:
        code = f.read()

    # 2. Убираем конфликты (HTTP vs HTTPS)
    if 'SECURE_PROXY_SSL_HEADER' not in code:
        code += "\nSECURE_PROXY_SSL_HEADER = ('HTTP_X_FORWARDED_PROTO', 'https')\n"

    # 3. Добавляем ВАШ адрес в доверенные
    your_host = "8000-m-s-use4a1-3qryt1t3yzu5u.us-east4.prod.colab.dev"

    # ALLOWED_HOSTS - разрешаем всё
    code = re.sub(r"ALLOWED_HOSTS\s*=\s*\[.*?\]", "ALLOWED_HOSTS = ['*']", code)

    # CSRF_TRUSTED_ORIGINS - доверяем вашему адресу
    csrf_line = f"CSRF_TRUSTED_ORIGINS = ['https://{your_host}']"
    if 'CSRF_TRUSTED_ORIGINS' in code:
        code = re.sub(r"CSRF_TRUSTED_ORIGINS\s*=\s*\[.*?\]", csrf_line, code)
    else:
        code += f"\n{csrf_line}\n"

    # 4. Отключаем проверку Referer (иногда именно она мешает в Colab)
    code += "\nCSRF_TRUSTED_ORIGINS = ['https://*.colab.dev']\n"

    with open(settings_path, 'w') as f:
        f.write(code)
    print("✅ Настройки обновлены!")

# 5. Перезапускаем сервер
os.system('pkill -f runserver')
time.sleep(1)

manage_path = 'manage.py' if os.path.exists('manage.py') else 'myproject/manage.py'

def run_django():
    os.system(f'python {manage_path} runserver 0.0.0.0:8000')

thread = threading.Thread(target=run_django)
thread.start()
time.sleep(5)

print("\n🔗 ССЫЛКА НА САЙТ:")
print(eval_js("google.colab.kernel.proxyPort(8000)"))

Найден файл настроек: myproject/settings.py
✅ Настройки обновлены!

🔗 ССЫЛКА НА САЙТ:
https://8000-m-s-use4a1-3qrty1t3yzu5u-a.us-east4.prod.r.colab.dev
